# Zein – Casein Fragments: Combined MEGADOCK Ranking

Aggregates results from the three per-ligand MEGADOCK notebooks —
`MEGADOCK_zein_beta_casein_docking.ipynb`, `MEGADOCK_zein_alphaS1_casein_docking.ipynb`,
`MEGADOCK_zein_kappa_casein_docking.ipynb` — into one side-by-side comparison table and chart,
so you don't have to eyeball three separate notebooks' outputs.

**Run this after** all three docking notebooks have completed (each of them copies a
`megadock_zein_{ligand}_results.zip` into your shared `zein_casein_docking/results` Drive folder
in its final "Package results for download" cell). This notebook doesn't re-run any docking — it
only reads what those three already produced, plus re-runs the (near-instant) `ppiscore` scoring
on each `.out` file so the score is captured into a value instead of just printed text.

**Two metrics, not one:** this notebook reports MEGADOCK's PPI E-score and the restraint-contact
fraction side by side, ranked separately. They measure different things (global FFT-search
likelihood vs. contact with your 40 specific target residues) and are not combined into a single
number beyond a simple average-of-ranks — with only 3 candidates, don't over-read small
differences in either metric.

## 1. Locate the three result archives in Drive

In [ ]:
# @title Mount Drive and check the three result zips exist
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_FOLDER = "/content/drive/MyDrive/UMD/Molecular_Docking/Megadocking/zein_casein_docking"
RESULTS_DIR = os.path.join(DRIVE_FOLDER, "results")

# ligand_name -> the file name each per-ligand notebook produced
LIGANDS = {
    "beta_casein":   "megadock_zein_beta_casein_results.zip",
    "alphaS1_casein": "megadock_zein_alphaS1_casein_results.zip",
    "kappa_casein":  "megadock_zein_kappa_casein_results.zip",
}

missing = [name for name, zipf in LIGANDS.items()
           if not os.path.exists(os.path.join(RESULTS_DIR, zipf))]

if missing:
    raise FileNotFoundError(
        f"Missing results for: {missing}. Run the corresponding "
        f"MEGADOCK_zein_{{ligand}}_docking.ipynb notebook (through its final \"Package results "
        f"for download\" cell) first -- that step is what copies the zip into "
        f"{RESULTS_DIR}."
    )

print("Found all 3 result archives in", RESULTS_DIR)


In [ ]:
# @title Unzip each ligand\'s results into its own local folder
import shutil

local_dirs = {}
for name, zipf in LIGANDS.items():
    dest = f"/content/results_{name}"
    shutil.unpack_archive(os.path.join(RESULTS_DIR, zipf), dest)
    local_dirs[name] = dest
    print(f"{name}: unzipped to {dest}")
    print("  contents:", sorted(os.listdir(dest)))


## 2. Re-score PPI E-value

Each per-ligand notebook printed its PPI score to the cell output but didn't save it to a file,
so it's re-computed here from the exported `dock_{ligand}.out` (this only reads the existing
docking output — it's a few seconds, not a re-run of the docking search).

In [ ]:
# @title Fetch the ppiscore script and run it for each ligand
import subprocess, re

if not os.path.exists("/content/ppiscore"):
    subprocess.run(["wget", "-q", "-O", "/content/ppiscore",
                     "https://raw.githubusercontent.com/akiyamalab/MEGADOCK/master/ppiscore"], check=True)
    os.chmod("/content/ppiscore", 0o755)

N_DECOYS = 10800  # -N used in all three docking notebooks

ppi_scores = {}
for name, d in local_dirs.items():
    outfile = os.path.join(d, f"dock_{name}.out")
    result = subprocess.run(["perl", "/content/ppiscore", outfile, str(N_DECOYS)],
                             capture_output=True, text=True, cwd=d)
    print(name, "->", result.stdout.strip() or result.stderr.strip())
    m = re.search(r"E = (-?[0-9.]+)", result.stdout)
    ppi_scores[name] = float(m.group(1)) if m else None

print()
print("PPI E-scores:", ppi_scores)


## 3. Restraint-satisfaction summary

Reads each ligand's `{ligand}_restraint_summary.csv` (written by the per-ligand notebook's
decoy-generation step) and pulls out the top-1 pose's restraint fraction plus the best and mean
fraction across its top-10 decoys.

In [ ]:
# @title Load each restraint_summary.csv
import pandas as pd

restraint_stats = {}
per_ligand_tables = {}
for name, d in local_dirs.items():
    csv_path = os.path.join(d, f"{name}_restraint_summary.csv")
    df = pd.read_csv(csv_path)
    per_ligand_tables[name] = df
    restraint_stats[name] = {
        "top1_fraction": df.loc[df["megadock_rank"] == 1, "fraction_satisfied"].iloc[0],
        "best_fraction_top10": df["fraction_satisfied"].max(),
        "mean_fraction_top10": df["fraction_satisfied"].mean(),
    }

pd.DataFrame(restraint_stats).T


## 4. Combine into one ranking table

In [ ]:
# @title Build the combined comparison table
combined = pd.DataFrame({
    "ligand": list(LIGANDS.keys()),
}).set_index("ligand")

combined["ppi_e_score"] = pd.Series(ppi_scores)
combined["top1_restraint_fraction"] = pd.Series({k: v["top1_fraction"] for k, v in restraint_stats.items()})
combined["best_restraint_fraction_top10"] = pd.Series({k: v["best_fraction_top10"] for k, v in restraint_stats.items()})
combined["mean_restraint_fraction_top10"] = pd.Series({k: v["mean_fraction_top10"] for k, v in restraint_stats.items()})

# Rank each metric separately (1 = best: highest PPI E-score, highest restraint fraction)
combined["rank_by_ppi_score"] = combined["ppi_e_score"].rank(ascending=False).astype(int)
combined["rank_by_restraint_top1"] = combined["top1_restraint_fraction"].rank(ascending=False).astype(int)
# Simple average-of-ranks as a rough combined ordering only -- with 3 candidates, treat as a tiebreaker
# hint, not a validated composite score.
combined["avg_rank"] = (combined["rank_by_ppi_score"] + combined["rank_by_restraint_top1"]) / 2

combined = combined.sort_values("avg_rank")
combined


## 5. Visualize

In [ ]:
# @title Bar charts: PPI E-score and top-1 restraint fraction, side by side
import matplotlib.pyplot as plt

order = combined.index  # already sorted by avg_rank
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].bar(order, combined.loc[order, "ppi_e_score"], color="#4C72B0")
axes[0].set_title("PPI E-score (higher = more distinctive vs. background)")
axes[0].set_ylabel("E-score")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(order, combined.loc[order, "top1_restraint_fraction"] * 100, color="#DD8452")
axes[1].set_title("Top-ranked pose: % restraint residues in contact")
axes[1].set_ylabel("% of 40 restraint residues")
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("/content/casein_combined_comparison.png", dpi=150)
plt.show()


## 6. Save and export

In [ ]:
# @title Save combined CSV + chart, copy back to Drive, download
from google.colab import files

combined.to_csv("/content/casein_combined_summary.csv")

combined_dir = os.path.join(RESULTS_DIR, "combined")
os.makedirs(combined_dir, exist_ok=True)
shutil.copy("/content/casein_combined_summary.csv", combined_dir)
shutil.copy("/content/casein_combined_comparison.png", combined_dir)
print(f"Saved combined summary + chart to {combined_dir}")

files.download("/content/casein_combined_summary.csv")
files.download("/content/casein_combined_comparison.png")


## Interpreting the combined table

- **`ppi_e_score`**: MEGADOCK's PPI score (Z-score of the top decoy against the score
  distribution of all N decoys for that pair). Higher / more positive is a stronger signal that
  the top-ranked pose stands out from the random background — treat as a coarse screening signal
  per the precision guidance in the individual notebooks, not a binding probability.
- **`top1_restraint_fraction`**: of your 40 zein restraint residues, the fraction in contact
  (within 5 Å) with the ligand in MEGADOCK's single best-scoring pose.
- **`best_/mean_restraint_fraction_top10`**: same, but the best and average across the top 10
  scored poses — useful if the #1 pose looks like an outlier relative to the rest of the top 10.
- **`rank_by_*` / `avg_rank`**: 1 = best on that metric. `avg_rank` is a simple mean of the two
  individual ranks, included as a quick eyeball ordering only — with 3 ligands being compared,
  don't treat small `avg_rank` differences as statistically meaningful. Look at the underlying
  `ppi_e_score` and restraint-fraction columns for the actual numbers.
- None of these are binding free energies. Use this table to decide which casein fragment(s) are
  worth following up on (e.g. with a slower/flexible refinement), not as a final answer.

## References

- Ohue M, et al. **MEGADOCK 4.0**: an ultra-high-performance protein-protein docking software for
  heterogeneous supercomputers. *Bioinformatics*, 30(22): 3281-3283, 2014.
  https://doi.org/10.1093/bioinformatics/btu532
- MEGADOCK GitHub: https://github.com/akiyamalab/MEGADOCK
- License: CC BY-NC 4.0 — non-commercial use only without authorization from Tokyo Institute of
  Technology.